# Task 2 — run every experiment in one Kaggle session

**Run this notebook once, unattended.** It trains the whole experiment plan, records every result, builds every submission candidate, and zips everything. No leaderboard submission is made; choose locally with `task2_results.ipynb`.

**On Kaggle**
1. *Settings → Accelerator → GPU T4 x2* (one GPU is used) and *Settings → Internet → On*.
2. *Save Version → **Save & Run All (Commit)***. It runs in the background; you can close the browser. GPU quota is used only while it runs (expected ≈ 2 h, hard stop for new training at `MAX_HOURS`).
3. When the version finishes, open its **Output** tab and download **`task2_results.zip`**.

**What the plan runs** (3 seeds each; every result is written to `results.csv` the moment it exists)

| stage | experiments |
|---|---|
| references | `E0_baseline`, `E0_none` (no external data — the required ablation), `E0_direct_level` (previous best) |
| fixed | `E1_exogtrend`, `E2a_features_only`, `E2_exogtrend_feat`, `E3_log` |
| adaptive | `E4_anchor`, `E5_anchor_direct` on the better of E2/E3; `E6_wider`, `E7_ctx336` on the best of E1–E5 |
| confirmation | 2 extra seeds (3, 4) for the top 3 experiments |
| outputs | a no-refit forecast for every experiment (average of its validation models) and an all-history refit forecast for the top 3 |

A failing step is logged in `plan_log.json` and the session continues, so a bug in one experiment does not waste the rest of the GPU time.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    # Clone outside /kaggle/working so the code and data are not copied into the outputs.
    REPO = "/tmp/msai_dl4stg"
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/rana-taqveem/msai_dl4stg.git {REPO}
    %cd "/tmp/msai_dl4stg/assignment_01/Question 2 - Leaderboard"
    RESULTS_ROOT = "/kaggle/working/task2_results"
    DEVICE, AMP = "cuda", True
else:
    # Running locally: stay in this folder and use whatever device is available.
    RESULTS_ROOT = "task2_results_local"
    DEVICE, AMP = ("cuda", True) if torch.cuda.is_available() else ("cpu", False)
!git log -1 --format="code version: %h %ci %s"

In [ ]:
MAX_HOURS = 6.0     # stop starting new training after this long (outputs are still built)
TRAIN_STRIDE = 2    # every 2nd training window; use 8 on CPU
SEEDS = [0, 1, 2]
EXTRA_SEEDS = [3, 4]
TOP_K = 3

### Smoke test (≈ 1 min): the whole pipeline on tiny settings, in a separate folder, so a code problem stops here instead of hours later.

In [ ]:
import run_all_experiments
run_all_experiments.main(["--root", RESULTS_ROOT + "_smoke", "--smoke", "--device", DEVICE]
                         + ([] if AMP else ["--no-amp"]))
import shutil
shutil.rmtree(RESULTS_ROOT + "_smoke", ignore_errors=True)
for leftover in (RESULTS_ROOT + "_smoke.zip",):
    if os.path.exists(leftover):
        os.remove(leftover)

### Full plan

In [ ]:
import importlib, task2_experiments
importlib.reload(task2_experiments); importlib.reload(run_all_experiments)
run_all_experiments.main(["--root", RESULTS_ROOT, "--device", DEVICE,
                          "--train-stride", str(TRAIN_STRIDE), "--max-hours", str(MAX_HOURS),
                          "--top-k", str(TOP_K), "--seeds", *map(str, SEEDS),
                          "--extra-seeds", *map(str, EXTRA_SEEDS)]
                         + ([] if AMP else ["--no-amp"]))

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", 40, "display.precision", 2)
cols = ["experiment", "seeds", "params", "RMSE_mean", "RMSE_std", "recent_RMSE_mean",
        "ens_RMSE", "ens_recent_RMSE", "MAE_mean", "sMAPE_mean", "best_epoch_mean"]
display(task2_experiments.compare()[cols])
display(task2_experiments.submissions())
print("download:", RESULTS_ROOT + ".zip")